# HumanEval Pass@k Evaluation Pipeline

This notebook evaluates HumanEval 10 problems per session,
resuming automatically from where it left off via Google Drive.

**How pass@k works here:**  
For each problem we generate N_SAMPLES = 10 independent completions.  
The official HumanEval unbiased estimator then computes pass@1, pass@4, and pass@8 simultaneously — no extra generation needed.

**Resume design**
Due to consumption of time when generating multiple codes answer, it may take a loog time that exceeds Colab's time limit, it can generate a subset of answers and then store them and resume them next time. Only evaluate after work throught all the problems.

## Step 1 · Install Dependencies

In [ ]:
# Clone the official HumanEval repo (provides the evaluator)
!git clone --quiet https://github.com/openai/human-eval
!pip install -q -e human-eval

# Core ML dependencies
!pip install -q transformers datasets accelerate bitsandbytes

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 41.1 MB/s eta 0:00:00


## Step 2 · Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Step 3 · Configuration

In [ ]:
# Import neccessary libraries
import os, json, torch, re
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

# direction and name of store file
DRIVE_DIR     = "/content/drive/MyDrive/deepseek_SFT_humaneval_pass8" # Edit the name as you like
SAMPLES_JSONL = os.path.join(DRIVE_DIR, "samples.jsonl")    # all completions (official format)
PROGRESS_FILE = os.path.join(DRIVE_DIR, "progress.json")    # resume pointer
SCORES_FILE   = os.path.join(DRIVE_DIR, "scores.json")      # running pass@k scores
# LoRA parameters for OLMo 2 7B SFT
LORA_PATH = "/content/drive/MyDrive/olmo2-gsm8k-grpo/checkpoint-450"

# Model configuration
MODEL_NAME     = "deepseek-ai/deepseek-math-7b-instruct"
# MODEL_NAME     = "deepseek-ai/deepseek-math-7b-rl"
# MODEL_NAME     = "allenai/OLMo-2-1124-7B-SFT"
PROBLEMS_PER_SESSION = 82   # how many new problems to tackle each run
N_SAMPLES      = 8          # must be larger than the k you want
MAX_NEW_TOKENS = 512        # The maximum length of generated tokens is 512
TEMPERATURE    = 0.8        # more diverse samples for pass@k
TOP_P          = 0.95

# Create the folder if it is not existed
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f"Drive directory : {DRIVE_DIR}")
print(f"GPU available   : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU             : {torch.cuda.get_device_name(0)}")

Drive directory : /content/drive/MyDrive/deepseek_SFT_humaneval_pass8
GPU available   : True
GPU             : NVIDIA A100-SXM4-80GB


## Step 4 · Load Dataset and Resume Progress

In [ ]:
# Download dataset
dataset     = load_dataset("openai_humaneval", split="test")
problems    = {row["task_id"]: row for row in dataset}
all_task_ids = list(problems.keys())   # 164 problems

print(f"Total HumanEval problems: {len(all_task_ids)}")

# Load previous progress
if os.path.exists(PROGRESS_FILE):
    with open(PROGRESS_FILE) as f:
        progress = json.load(f)
    completed_ids = set(progress["completed_task_ids"])
    print(f"Resuming — {len(completed_ids)} / {len(all_task_ids)} problems already done.")
else:
    completed_ids = set()
    print("Starting fresh.")

remaining_ids = [tid for tid in all_task_ids if tid not in completed_ids]
batch_ids     = remaining_ids[:PROBLEMS_PER_SESSION]

if batch_ids:
    print(f"This session    : {len(batch_ids)} problems  ({batch_ids[0]} … {batch_ids[-1]})")
    print(f"Completions to generate: {len(batch_ids)} × {N_SAMPLES} = {len(batch_ids)*N_SAMPLES}")
else:
    print("✅ All problems are done! Jump to Step 7 to see final scores.")

Total HumanEval problems: 164
Resuming — 82 / 164 problems already done.
This session    : 82 problems  (HumanEval/82 … HumanEval/163)
Completions to generate: 82 × 8 = 656


## Step 5 · Load Model  
Sometime the RAM is not big enough for loading the model, so I quantitize the model into 4 bit, specially when A100 is not available.

In [ ]:
# Setup the configuration for quantitizing
if not batch_ids:
    print("No batch to run — skipping model load.")
else:
    print(f"Loading {MODEL_NAME} in 4-bit …")
    quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

# Load the model and tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=quant_config,
    device_map="auto",
    dtype=torch.float16,
)

# Uncomment this when you want to use the OLMo-2-1124-7B-SFT with LoRA parameter
# model = PeftModel.from_pretrained(base_model, LORA_PATH)
model.eval()
print("Model ready.")

Loading deepseek-ai/deepseek-math-7b-instruct in 4-bit …


config.json:   0%|          | 0.00/594 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

pytorch_model.bin.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Loading weights:   0%|          | 0/273 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/121 [00:00<?, ?B/s]

Model ready.


## Step 6 · Generate N Completions per Problem & Append to samples.jsonl

In [ ]:
# Helping function for prompt and extract the answers
def build_prompt(problem: dict) -> str:
  """Build the prompt for humanEval"""
    return (
        "Complete the following Python function. "
        "Output ONLY the function body — no explanation, no markdown.\n\n"
        + problem["prompt"]
    )

def extract_completion(generated_text: str, prompt: str) -> str:
    """Strip the echoed prompt; keep only the new tokens."""
    if generated_text.startswith(prompt):
        return generated_text[len(prompt):]
    # Fallback: find the function body after the last def line in the prompt
    return generated_text

# Generate multiple completions
@torch.inference_mode()
def generate_n_completions(problem: dict, n: int) -> list[str]:
    """Generate n independent completions for one problem."""
    prompt   = build_prompt(problem)
    inputs   = tokenizer(prompt, return_tensors="pt").to(model.device)
    input_len = inputs["input_ids"].shape[1]

    # Generate n samples in one batched call
    output_ids = model.generate(
        **inputs,
        max_new_tokens=MAX_NEW_TOKENS,
        temperature=TEMPERATURE,
        top_p=TOP_P,
        do_sample=True,
        num_return_sequences=n,
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

    completions = []
    for seq in output_ids:
        # Decode only the newly generated tokens
        new_tokens = seq[input_len:]
        completion = tokenizer.decode(new_tokens, skip_special_tokens=True)
        completions.append(completion)
    return completions


if batch_ids:
    # Append new samples to the running JSONL file
    with open(SAMPLES_JSONL, "a") as out_f:
        for i, task_id in enumerate(batch_ids):
            print(f"[{i+1:02d}/{len(batch_ids)}] {task_id} — generating {N_SAMPLES} samples …", end=" ")
            completions = generate_n_completions(problems[task_id], N_SAMPLES)
            for comp in completions:
                out_f.write(json.dumps({"task_id": task_id, "completion": comp}) + "\n")
            print("done")

    # Update progress
    completed_ids.update(batch_ids)
    with open(PROGRESS_FILE, "w") as f:
        json.dump({"completed_task_ids": list(completed_ids)}, f, indent=2)

    print(f"\nsamples.jsonl updated  → {SAMPLES_JSONL}")
    print(f"Progress saved         → {len(completed_ids)} / {len(all_task_ids)} problems done")

[01/82] HumanEval/82 — generating 8 samples … done
[02/82] HumanEval/83 — generating 8 samples … done
[03/82] HumanEval/84 — generating 8 samples … done
[04/82] HumanEval/85 — generating 8 samples … done
[05/82] HumanEval/86 — generating 8 samples … done
[06/82] HumanEval/87 — generating 8 samples … done
[07/82] HumanEval/88 — generating 8 samples … done
[08/82] HumanEval/89 — generating 8 samples … done
[09/82] HumanEval/90 — generating 8 samples … done
[10/82] HumanEval/91 — generating 8 samples … done
[11/82] HumanEval/92 — generating 8 samples … done
[12/82] HumanEval/93 — generating 8 samples … done
[13/82] HumanEval/94 — generating 8 samples … done
[14/82] HumanEval/95 — generating 8 samples … done
[15/82] HumanEval/96 — generating 8 samples … done
[16/82] HumanEval/97 — generating 8 samples … done
[17/82] HumanEval/98 — generating 8 samples … done
[18/82] HumanEval/99 — generating 8 samples … done
[19/82] HumanEval/100 — generating 8 samples … done
[20/82] HumanEval/101 — genera

## Step 7 · Evaluate with the Official HumanEval Evaluator

This calls `evaluate_functional_correctness` directly from Python and reports pass@1, pass@4, pass@8.

After generate all the complements, uncomment the following cells.

In [ ]:
# import sys
# sys.path.insert(0, "/content/human-eval")

# import human_eval
# print(human_eval.__file__)

/content/human-eval/human_eval/__init__.py


In [ ]:
# import os
# os.environ["HF_ALLOW_CODE_EVAL"] = "1"   # required by some evaluator versions

# from human_eval.evaluation import evaluate_functional_correctness

# if not os.path.exists(SAMPLES_JSONL):
#     print("No samples found yet — run Steps 5-6 first.")
# else:
#     # Count how many samples are in the file
#     with open(SAMPLES_JSONL) as f:
#         n_rows = sum(1 for _ in f)
#     n_problems_in_file = len(completed_ids)
#     print(f"Evaluating {n_rows} completions across {n_problems_in_file} problems …")
#     print(f"(k values: 1, 4, 8  —  n samples per problem: {N_SAMPLES})")
#     print()

#     # Official evaluator — runs tests in sandboxed subprocesses
#     # n_workers controls parallelism; 4 is safe on Colab
#     scores = evaluate_functional_correctness(
#         sample_file=SAMPLES_JSONL,
#         k=[1, 4, 8],
#         n_workers=4,
#         timeout=10.0,
#     )

#     print("\n" + "═"*45)
#     print(" deepseek-ai/deepseek-math-7b-instruct  ·  HumanEval Results")
#     print(f" Problems evaluated : {n_problems_in_file} / {len(all_task_ids)}")
#     print("═"*45)
#     for metric, value in sorted(scores.items()):
#         print(f"  {metric:<10} {value:.4f}  ({value*100:.2f}%)")
#     print("═"*45)

#     # Persist scores
#     scores["problems_evaluated"] = n_problems_in_file
#     scores["total_problems"]     = len(all_task_ids)
#     with open(SCORES_FILE, "w") as f:
#         json.dump(scores, f, indent=2)
#     print(f"\nScores saved → {SCORES_FILE}")

#     if n_problems_in_file == len(all_task_ids):
#         print("\n Full evaluation complete!")

Evaluating 1312 completions across 164 problems …
(k values: 1, 4, 8  —  n samples per problem: 8)

Reading samples...


1312it [00:00, 8965.01it/s]


Running test suites...


100%|██████████| 1312/1312 [01:32<00:00, 14.22it/s]


Writing results to /content/drive/MyDrive/deepseek_SFT_humaneval_pass8/samples.jsonl_results.jsonl...


100%|██████████| 1312/1312 [00:00<00:00, 58337.59it/s]


═════════════════════════════════════════════
 deepseek-ai/deepseek-math-7b-instruct  ·  HumanEval Results
 Problems evaluated : 164 / 164
═════════════════════════════════════════════
  pass@1     0.4459  (44.59%)
  pass@4     0.6547  (65.47%)
  pass@8     0.7256  (72.56%)
═════════════════════════════════════════════

Scores saved → /content/drive/MyDrive/deepseek_SFT_humaneval_pass8/scores.json

🎉 Full evaluation complete!
